# 01 - HydroServer Things and Metadata

In [ ]:
!pip install hydroserverpy --quiet

In [2]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from datetime import datetime, timezone

from __future__ import annotations
from typing import Any, Dict, List, Optional

## Helper Functions

In [3]:
def _prompt_if_needed(value, prompt):
    return value if value else getpass(prompt)

def resource_uid(resource):
    """
    Extract a unique identifier from a HydroServer resource object, dictionary, or string.
    """
    if resource is None:
        return None
    if isinstance(resource, str):
        return resource
    if isinstance(resource, dict):
        for key in ("uid", "id", "workspace_id"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("uid", "id", "workspace_id"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_uid(dumped)
    return None


def resource_name(resource):
    """
    Find a human-readable name/code for a HydroServer resource.
    """
    if resource is None:
        return None
    if isinstance(resource, dict):
        for key in ("name", "code", "definition", "sampling_feature_code", "symbol"):
            if resource.get(key):
                return str(resource[key])
        return None
    for attr in ("name", "code", "definition", "sampling_feature_code", "symbol"):
        value = getattr(resource, attr, None)
        if value:
            return str(value)
    if hasattr(resource, "model_dump"):
        dumped = resource.model_dump()
        if isinstance(dumped, dict):
            return resource_name(dumped)
    return type(resource).__name__


def collection_items(collection):
    """
    Normalize a hydroserverpy collection object into a plain list.
    """
    return list(getattr(collection, "items", collection) or [])


def list_workspace_resources(endpoint_name, *, page_size=1000):
    """
    List all resources of one endpoint type that belong to the selected workspace.
    Example endpoint_name values:
      things, datastreams, observedproperties, units, sensors,
      processinglevels, resultqualifiers, orchestrationsystems,
      dataconnections, tasks
    """
    if hs_api is None or workspace_uid is None:
        return []

    endpoint = getattr(hs_api, endpoint_name)
    collection = endpoint.list(
        workspace=workspace_uid,
        fetch_all=True,
        page_size=page_size,
    )
    return collection_items(collection)


def matches_resource(resource, lookup):
    """
    Check whether a resource matches all lookup fields.
    """
    for key, expected in lookup.items():
        actual = getattr(resource, key, None)
        if actual is None and hasattr(resource, "model_dump"):
            actual = resource.model_dump().get(key)
        if str(actual) != str(expected):
            return False
    return True


def find_workspace_resource(endpoint_name, lookup):
    """
    Find one resource in the current workspace using attribute equality.
    """
    for resource in list_workspace_resources(endpoint_name):
        if matches_resource(resource, lookup):
            return resource
    return None


def get_or_create_workspace_resource(endpoint_name, lookup, create_kwargs, *, label=None):
    """
    Idempotent create helper:
    - First looks in the workspace for an existing resource matching `lookup`
    - If found, returns it
    - Otherwise creates it with `create_kwargs`

    This keeps the notebook rerunnable without a separate created_resources registry.
    """
    label = label or endpoint_name
    existing = find_workspace_resource(endpoint_name, lookup)

    if existing is not None:
        print(f"Using existing {label}: {resource_name(existing)} | uuid={resource_uid(existing)}")
        return existing

    created_resource = getattr(hs_api, endpoint_name).create(**create_kwargs)
    print(f"Created {label}: {resource_name(created_resource)} | uuid={resource_uid(created_resource)}")
    return created_resource


def workspace_thing_map():
    """
    Build a station_id -> Thing object map directly from the workspace.
    """
    return {
        str(getattr(thing, "sampling_feature_code", "")): thing
        for thing in list_workspace_resources("things")
        if getattr(thing, "sampling_feature_code", None)
    }


def workspace_inventory_dataframe():
    """
    Summarize the resources currently in the selected workspace.
    """
    rows = []
    for label, endpoint_name in [
        ("things", "things"),
        ("datastreams", "datastreams"),
        ("observed_properties", "observedproperties"),
        ("units", "units"),
        ("sensors", "sensors"),
        ("processing_levels", "processinglevels"),
        ("result_qualifiers", "resultqualifiers"),
        ("orchestration_systems", "orchestrationsystems"),
        ("data_connections", "dataconnections"),
        ("tasks", "tasks"),
    ]:
        try:
            resources = list_workspace_resources(endpoint_name)
            rows.append({
                "resource_type": label,
                "count": len(resources),
                "examples": ", ".join(resource_name(item) or "" for item in resources[:5]),
            })
        except Exception as exc:
            rows.append({
                "resource_type": label,
                "count": None,
                "examples": f"Could not inspect: {exc}",
            })
    return pd.DataFrame(rows)

def optional_float(value, default=None):
    if pd.isna(value):
        return default
    return float(value)

In [4]:
def identifier_text(value):
    if pd.isna(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)

def station_file_index(directory):
    directory = Path(directory)
    if not directory.exists():
        return {}
    return {path.stem: path for path in directory.glob("*.csv")}

def resolve_station_file(index, station, identifiers):
    for identifier in identifiers:
        key = identifier_text(identifier)
        if key and key in index:
            return index[key]
    return None


def normalize_hydroweb_water_level(path, station):
    frame = pd.read_csv(path)
    if list(frame.columns) != ["Datetime", "Water Level (m)"]:
        raise ValueError(f"{path.name} should have columns: Datetime, Water Level (m)")
    return pd.DataFrame({
        "phenomenon_time": pd.to_datetime(frame["Datetime"], utc=True),
        "result": pd.to_numeric(frame["Water Level (m)"], errors="raise"),
        "source": "Hydroweb/Theia",
        "source_identifier": identifier_text(station["COMID_v1"]),
        "station_id": station["ID"],
        "observed_property": "Water Level",
        "unit": "m",
    }).dropna(subset=["phenomenon_time", "result"]).reset_index(drop=True)


def normalize_two_column_geoglows(path, station):
    frame = pd.read_csv(path)
    if len(frame.columns) != 2:
        raise ValueError(f"{path.name} should have exactly two columns")
    timestamp_column, value_column = frame.columns
    return pd.DataFrame({
        "phenomenon_time": pd.to_datetime(frame[timestamp_column], utc=True),
        "result": pd.to_numeric(frame[value_column], errors="raise"),
        "source": "GEOGLOWS",
        "source_identifier": identifier_text(station["COMID_v2"]),
        "station_id": station["ID"],
        "observed_property": "Streamflow",
        "unit": "m3/s",
    }).dropna(subset=["phenomenon_time", "result"]).reset_index(drop=True)

In [5]:
# -----------------------------------------------------------------------------
# Helper 1: Get all datastreams from the workspace
# -----------------------------------------------------------------------------
#
# This function returns a dictionary where:
#   key   = datastream name
#   value = HydroServer datastream object
#
# Example:
#   {
#       "Workshop 12345 Hydroweb Water Level": <Datastream object>,
#       "Workshop 12345 GEOGLOWS Streamflow": <Datastream object>,
#   }
#
# We use datastream names because the notebook created predictable names earlier.
# -----------------------------------------------------------------------------

def workspace_datastream_map():
    """
    Get all datastreams in the current HydroServer workspace.

    Returns
    -------
    dict
        Dictionary mapping datastream name to datastream object.
    """
    datastreams = list_workspace_resources("datastreams")

    return {
        resource_name(datastream): datastream
        for datastream in datastreams
    }


# -----------------------------------------------------------------------------
# Helper 2: Prepare observations for HydroServer
# -----------------------------------------------------------------------------
#
# Different data sources often use different column names and date formats.
# Earlier in the notebook, we created normalizer functions that convert the
# source files into a common format.
#
# This function performs a final cleanup before uploading:
#
#   1. Keep only the required HydroServer columns.
#   2. Convert times to timezone-aware UTC timestamps.
#   3. Convert results to numeric values.
#   4. Drop invalid rows.
#   5. Sort observations by time.
#   6. Remove duplicate timestamps.
#
# HydroServer requires clean timestamps and numeric results, so doing this
# before upload prevents many common errors.
# -----------------------------------------------------------------------------

def prepare_observations_for_hydroserver(observations_df):
    """
    Prepare a DataFrame for HydroServer observation upload.

    Parameters
    ----------
    observations_df : pandas.DataFrame
        DataFrame with at least:
        - phenomenon_time
        - result

    Returns
    -------
    pandas.DataFrame
        Cleaned DataFrame ready for datastream.load_observations().
    """

    required_columns = ["phenomenon_time", "result"]

    missing_columns = [
        column
        for column in required_columns
        if column not in observations_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Observation DataFrame is missing required columns: {missing_columns}"
        )

    payload = observations_df[required_columns].copy()

    # Convert observation times to timezone-aware UTC datetimes.
    payload["phenomenon_time"] = pd.to_datetime(
        payload["phenomenon_time"],
        utc=True,
        errors="raise",
    )

    # Convert observation values to numbers.
    payload["result"] = pd.to_numeric(
        payload["result"],
        errors="raise",
    )

    # Remove invalid rows, sort by time, and avoid duplicate timestamps.
    payload = (
        payload
        .dropna(subset=["phenomenon_time", "result"])
        .sort_values("phenomenon_time")
        .drop_duplicates(subset=["phenomenon_time"], keep="last")
        .reset_index(drop=True)
    )

    return payload


# -----------------------------------------------------------------------------
# Helper 3: Upload one observation table to one datastream
# -----------------------------------------------------------------------------
#
# This function uploads one cleaned observation table to one HydroServer
# datastream.
#
# If UPLOAD_MODE is "replace":
#   Existing observations in the datastream are replaced.
#
# Otherwise:
#   Observations are uploaded using the default hydroserverpy behavior.
# -----------------------------------------------------------------------------

def upload_observations_to_datastream(datastream, observations_df, *, mode="replace"):
    """
    Upload observations to one HydroServer datastream.

    Parameters
    ----------
    datastream
        HydroServer datastream object.

    observations_df : pandas.DataFrame
        Observations to upload.

    mode : str
        Use "replace" for rerunnable workshop notebooks.

    Returns
    -------
    dict
        Upload status information.
    """

    payload = prepare_observations_for_hydroserver(observations_df)

    if payload.empty:
        return {
            "status": "skipped",
            "rows_uploaded": 0,
            "message": "No valid observations found after cleaning.",
        }

    if mode == "replace":
        datastream.load_observations(payload, mode="replace")
    else:
        datastream.load_observations(payload)

    return {
        "status": "uploaded",
        "rows_uploaded": len(payload),
        "begin_time": payload["phenomenon_time"].min(),
        "end_time": payload["phenomenon_time"].max(),
        "message": "",
    }

In [7]:
def _obj_id(obj: Any) -> str:
  return str(getattr(obj, "uid", None) or getattr(obj, "id", None) or getattr(obj, "name", None) or repr(obj))

def _obj_label(obj: Any) -> str:
  name = getattr(obj, "name", None)
  code = getattr(obj, "code", None)
  uid = getattr(obj, "uid", None) or getattr(obj, "id", None)

  if name and uid:
    return f"{name} ({uid})"
  if code and uid:
    return f"{code} ({uid})"
  return _obj_id(obj)

def _record_deleted(kind: str, obj: Any) -> None:
  summary["deleted"].setdefault(kind, []).append(_obj_label(obj))

def _record_error(kind: str, obj: Any, exc: Exception) -> None:
  summary["errors"].append({"kind": kind, "resource": _obj_label(obj), "error": repr(exc),})
  if not continue_on_error:
    raise exc

def _list(resource_name: str) -> List[Any]:
  resource = getattr(hs_api, resource_name)
  collection = resource.list(workspace=workspace_uid, fetch_all=True, page_size=page_size,)
  return list(getattr(collection, "items", collection))

def _delete_objects(kind: str, objects: List[Any]) -> None:
  for obj in objects:
    try:
      if not dry_run:
        obj.delete()
        _record_deleted(kind, obj)
    except Exception as exc:
      _record_error(kind, obj, exc)

## Setup and Creation Controls
Configure the workspace, authentication, and safe cleanup behavior before creating anything.

In [8]:
try:
    from hydroserverpy import HydroServer
except Exception as exc:
    HydroServer = None
    print(f"hydroserverpy is not available yet: {exc}")

HYDROSERVER_HOST = "https://playground.hydroserver.org"
WORKSPACE_NAME = "demo1"

AUTH_METHOD = "password"
HYDROSERVER_API_KEY = ""

WORKSPACE_IS_PRIVATE = False
CREATE_WORKSPACE_IF_MISSING = True
DELETE_CREATED_RESOURCES_AT_END = True
SELECTED_STATION_CSV="/content/sample_data/Uganda_Hydroweb_subset.csv"
DEMO_RESOURCE_PREFIX = "Uganda Demo"
DEMO_RUN_SUFFIX = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S") #generates a string like 20240428004919 (YearMonthDayHourMinuteSecond) which is useful for creating unique identifiers for resources

print(f"HydroServer host: {HYDROSERVER_HOST}")
print(f"Workspace: {WORKSPACE_NAME}")
print(f"Authentication mode: {AUTH_METHOD}")
print(f"Data Station: {SELECTED_STATION_CSV}")

HydroServer host: https://playground.hydroserver.org
Workspace: demo1
Authentication mode: password
Data Station: /content/sample_data/Uganda_Hydroweb_subset.csv


## Connect to HydroServer

In [9]:
hs_api = None

if HydroServer is None:
    print("Install hydroserverpy before connecting to HydroServer.")
elif AUTH_METHOD == "anonymous":
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST)
        print("Connected anonymously. Anonymous mode can read public data but cannot create or upload resources.")
    except Exception as exc:
        print(f"Anonymous connection failed: {exc}")
elif AUTH_METHOD == "password":
    try:
        username = input("HydroServer username: ")
        password = getpass("HydroServer password: ")
        hs_api = HydroServer(host=HYDROSERVER_HOST, email=username, password=password)
        print("Connected with password authentication.")
    except Exception as exc:
        print(f"Password connection failed: {exc}")
elif AUTH_METHOD == "api_key":
    api_key = _prompt_if_needed(HYDROSERVER_API_KEY, "HydroServer API key: ")
    try:
        hs_api = HydroServer(host=HYDROSERVER_HOST, apikey=api_key)
        print("Connected with API-key authentication.")
    except Exception as exc:
        print(f"API-key connection failed: {exc}")
else:
    raise ValueError("AUTH_METHOD must be 'anonymous' or 'api_key'.")

HydroServer username: jorgessanchez7@gmail.com
HydroServer password: ··········
Connected with password authentication.


## Find or Optionally Create Demo Workspace

In [10]:
workspace = None
workspace_uid = None

if hs_api is None:
    print("Skipping workspace lookup because the HydroServer client is unavailable.")
elif AUTH_METHOD == "anonymous":
    print(f"Anonymous mode: cannot create or manage workspace '{WORKSPACE_NAME}'.")
    print("Switch AUTH_METHOD to 'password' or 'api_key' for live creation/upload demos.")
else:
    try:
        workspaces = hs_api.workspaces.list(fetch_all=True, is_associated=True)
        workspace_items = collection_items(workspaces)
        workspace = next((item for item in workspace_items if getattr(item, "name", None) == WORKSPACE_NAME), None)

        if workspace is None and CREATE_WORKSPACE_IF_MISSING:
            workspace = hs_api.workspaces.create(name=WORKSPACE_NAME, is_private=WORKSPACE_IS_PRIVATE)
            print(f"Created workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")
        elif workspace is None:
            print(f"Workspace '{WORKSPACE_NAME}' was not found. Ask the facilitator to create it first.")
        else:
            print(f"Using workspace: {getattr(workspace, 'name', WORKSPACE_NAME)}")

        workspace_uid = resource_uid(workspace)
        if workspace_uid:
            print(f"Workspace UUID: {workspace_uid}")

    except Exception as exc:
        print(f"Could not find or create workspace '{WORKSPACE_NAME}': {exc}")


Created workspace: demo1
Workspace UUID: 019dd255-f7ac-7e50-a305-8ffcbdc1d236


## Prepare Thing and Metadata Templates

In [11]:
stations_df = pd.read_csv(SELECTED_STATION_CSV)
stations = stations_df.to_dict(orient='records')
demo_resource_prefix = f"{DEMO_RESOURCE_PREFIX} {DEMO_RUN_SUFFIX}"

def build_things_template(stations):
    stations_template = []
    for station in stations:

      stations_template.append({
          "name": f"{station.get('Name', 'Uganda Station')}",
          "description": "Uganda Hydroweb/GEOGLOWS station used for workshop observations.",
          "sampling_feature_type": "Site",
          "sampling_feature_code": str(station["ID"]),
          "site_type": "Stream",
          "latitude": optional_float(station["Latitude"]),
          "longitude": optional_float(station["Longitude"]),
          "elevation_m": optional_float(station.get("Elevation"), 0.0),
          "elevation_datum": str(station.get("Ellipsoid", "WGS84")),
          "state": "",
          "county": str(station.get("River", "")),
          "country": "UG",
          "data_disclaimer": "Workshop demonstration data from local Hydroweb and GEOGLOWS CSV files.",
          "is_private": False,
          "workspace": workspace_uid or "<workspace-uuid>",
      })
    return stations_template

water_level_observed_property_template = {
    "name": f"{demo_resource_prefix} Water Level",
    "definition": "Satellite altimetry water level",
    "description": "Hydroweb/Theia water level for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"WaterLevel_{DEMO_RUN_SUFFIX}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_observed_property_template = {
    "name": f"{demo_resource_prefix} Streamflow",
    "definition": "Water discharge in a river channel",
    "description": "GEOGLOWS streamflow for Uganda stations.",
    "observed_property_type": "Hydrology",
    "code": f"Streamflow_{DEMO_RUN_SUFFIX}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
water_level_unit_template = {
    "name": f"{demo_resource_prefix} Meter",
    "symbol": "m",
    "definition": "Meter",
    "unit_type": "Length",
    "workspace": workspace_uid or "<workspace-uuid>",
}
streamflow_unit_template = {
    "name": f"{demo_resource_prefix} Cubic meters per second",
    "symbol": "m3/s",
    "definition": "Cubic meters per second",
    "unit_type": "Discharge",
    "workspace": workspace_uid or "<workspace-uuid>",
}
hydroweb_sensor_template = {
    "name": f"{demo_resource_prefix} Hydroweb Theia Altimetry",
    "description": "Hydroweb/Theia satellite altimetry water-level source.",
    "encoding_type": "application/json",
    "manufacturer": "Theia Hydroweb",
    "sensor_model": str(stations[0].get("Missions", "Hydroweb")),
    "sensor_model_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_type": "Satellite altimetry",
    "method_link": "https://catalogue.theia.data-terra.org/collection/HYDROWEB_RIVERS_OPE",
    "method_code": f"HYDROWEB_{DEMO_RUN_SUFFIX}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
geoglows_sensor_template = {
    "name": f"{demo_resource_prefix} GEOGLOWS RFS",
    "description": "GEOGLOWS modeled streamflow source.",
    "encoding_type": "application/json",
    "manufacturer": "GEOGLOWS",
    "sensor_model": "GEOGLOWS RFS",
    "sensor_model_link": "https://data.geoglows.org/",
    "method_type": "Model",
    "method_link": "https://data.geoglows.org/",
    "method_code": f"GEOGLOWS_{DEMO_RUN_SUFFIX}",
    "workspace": workspace_uid or "<workspace-uuid>",
}
processing_level_template = {
    "code": f"RAW_{DEMO_RUN_SUFFIX}",
    "definition": "Raw",
    "explanation": "Data have not been processed or quality controlled in the workshop.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
result_qualifier_template = {
    "code": f"SUSPECT_{DEMO_RUN_SUFFIX}",
    "description": "Observation should be reviewed before operational use.",
    "workspace": workspace_uid or "<workspace-uuid>",
}
metadata_templates = pd.DataFrame([
    {"resource_type": "things", "name_or_code": ", ".join([s.get('Name', '') for s in stations])},
    {"resource_type": "observed_property", "name_or_code": water_level_observed_property_template["code"]},
    {"resource_type": "observed_property", "name_or_code": streamflow_observed_property_template["code"]},
    {"resource_type": "unit", "name_or_code": water_level_unit_template["symbol"]},
    {"resource_type": "unit", "name_or_code": streamflow_unit_template["symbol"]},
    {"resource_type": "sensor", "name_or_code": hydroweb_sensor_template["method_code"]},
    {"resource_type": "sensor", "name_or_code": geoglows_sensor_template["method_code"]},
    {"resource_type": "processing_level", "name_or_code": processing_level_template["code"]},
    {"resource_type": "result_qualifier", "name_or_code": result_qualifier_template["code"]},
])



display(metadata_templates)

,resource_type,name_or_code
0,things,"Nile_achwa_km5268, Nile_akagera_km6130, Nile_a..."
1,observed_property,WaterLevel_20260428042435
2,observed_property,Streamflow_20260428042435
3,unit,m
4,unit,m3/s
5,sensor,HYDROWEB_20260428042435
6,sensor,GEOGLOWS_20260428042435
7,processing_level,RAW_20260428042435
8,result_qualifier,SUSPECT_20260428042435


## Create Needed Metadata and an Example Thing

In [12]:
water_level_observed_property = None
streamflow_observed_property = None
water_level_unit = None
streamflow_unit = None
hydroweb_sensor = None
geoglows_sensor = None
processing_level = None
result_qualifier = None
workspace_things = []

if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping creation because an authenticated workspace is required.")
else:
    try:
        water_level_observed_property = get_or_create_workspace_resource(
            "observedproperties",
            {"code": water_level_observed_property_template["code"]},
            water_level_observed_property_template,
            label="observed property",
        )
        streamflow_observed_property = get_or_create_workspace_resource(
            "observedproperties",
            {"code": streamflow_observed_property_template["code"]},
            streamflow_observed_property_template,
            label="observed property",
        )
        water_level_unit = get_or_create_workspace_resource(
            "units",
            {"name": water_level_unit_template["name"]},
            water_level_unit_template,
            label="unit",
        )
        streamflow_unit = get_or_create_workspace_resource(
            "units",
            {"name": streamflow_unit_template["name"]},
            streamflow_unit_template,
            label="unit",
        )
        hydroweb_sensor = get_or_create_workspace_resource(
            "sensors",
            {"method_code": hydroweb_sensor_template["method_code"]},
            hydroweb_sensor_template,
            label="sensor",
        )
        geoglows_sensor = get_or_create_workspace_resource(
            "sensors",
            {"method_code": geoglows_sensor_template["method_code"]},
            geoglows_sensor_template,
            label="sensor",
        )
        processing_level = get_or_create_workspace_resource(
            "processinglevels",
            {"code": processing_level_template["code"]},
            processing_level_template,
            label="processing level",
        )
        result_qualifier = get_or_create_workspace_resource(
            "resultqualifiers",
            {"code": result_qualifier_template["code"]},
            result_qualifier_template,
            label="result qualifier",
        )

        for thing_template in build_things_template(stations):
            thing = get_or_create_workspace_resource(
                "things",
                {"sampling_feature_code": str(thing_template["sampling_feature_code"])},
                thing_template,
                label="thing",
            )
            workspace_things.append(thing)

    except Exception as exc:
        print(f"Could not create or retrieve metadata resources: {exc}")

display(workspace_inventory_dataframe())


Created observed property: Uganda Demo 20260428042435 Water Level | uuid=019dd257-adb4-7792-b27b-687a511bb9bc
Created observed property: Uganda Demo 20260428042435 Streamflow | uuid=019dd257-b20f-7f0b-8965-dea7e601f014
Created unit: Uganda Demo 20260428042435 Meter | uuid=019dd257-b92b-7494-80e7-d393fa5cb624
Created unit: Uganda Demo 20260428042435 Cubic meters per second | uuid=019dd257-bd9a-7110-9d59-a791b537e179
Created sensor: Uganda Demo 20260428042435 Hydroweb Theia Altimetry | uuid=019dd257-c20d-75e4-909d-28d31f870e35
Created sensor: Uganda Demo 20260428042435 GEOGLOWS RFS | uuid=019dd257-c91d-7239-badc-466a83b5f1ea
Created processing level: RAW_20260428042435 | uuid=019dd257-cdab-7225-83a6-989b15ce3372
Created result qualifier: SUSPECT_20260428042435 | uuid=019dd257-d223-771a-939c-fb90c05be169
Created thing: Nile_achwa_km5268 | uuid=019dd257-d6b1-748c-baf3-aed0b33f4cf7
Created thing: Nile_akagera_km6130 | uuid=019dd257-ddf8-763a-b1ea-8921cf534ea0
Created thing: Nile_akagera_km6

,resource_type,count,examples
0,things,5,"Nile_achwa_km5268, Nile_akagera_km6130, Nile_a..."
1,datastreams,0,
2,observed_properties,2,"Uganda Demo 20260428042435 Water Level, Uganda..."
3,units,2,"Uganda Demo 20260428042435 Meter, Uganda Demo ..."
4,sensors,2,Uganda Demo 20260428042435 Hydroweb Theia Alti...
5,processing_levels,1,RAW_20260428042435
6,result_qualifiers,1,SUSPECT_20260428042435
7,orchestration_systems,0,
8,data_connections,0,
9,tasks,0,


## Inspect Workspace Metadata

In [13]:
if hs_api is None:
    print("Skipping live metadata inspection because the HydroServer client is unavailable.")
else:
    for label, endpoint_name in [
        ("things", "things"),
        ("datastreams", "datastreams"),
        ("observed properties", "observedproperties"),
        ("units", "units"),
        ("sensors", "sensors"),
    ]:
        try:
            endpoint = getattr(hs_api, endpoint_name)
            collection = endpoint.list(workspace=workspace_uid)
            items = getattr(collection, "items", collection)
            print(f"{label}: {len(list(items)) if not isinstance(items, list) else len(items)} records")
        except Exception as exc:
            print(f"Could not inspect {label}: {exc}")

things: 5 records
datastreams: 0 records
observed properties: 2 records
units: 2 records
sensors: 2 records


## Prepare some Time Series

In [14]:
#Define locations for the folders
HYDROWEB_DIR="/content/sample_data/ts/hydroweb"
GEOGLOWS_DIR="/content/sample_data/ts/geoglows"

In [16]:
hydroweb_index = station_file_index(HYDROWEB_DIR)
geoglows_index = station_file_index(GEOGLOWS_DIR)

bulk_rows = []
for _, station in stations_df.iterrows():
    hydroweb_file = resolve_station_file(hydroweb_index, station, [station["ID"], station["COMID_v1"]])
    geoglows_file = resolve_station_file(geoglows_index, station, [station["ID"], station["COMID_v2"]])
    has_comids = identifier_text(station["COMID_v1"]) not in ("", "0") and identifier_text(station["COMID_v2"]) not in ("", "0")
    skip_reasons = []
    if not has_comids:
        skip_reasons.append("missing COMID_v1 or COMID_v2")
    if hydroweb_file is None:
        skip_reasons.append("missing Hydroweb CSV")
    if geoglows_file is None:
        skip_reasons.append("missing GEOGLOWS CSV")
    bulk_rows.append({
        "station_id": station["ID"],
        "name": station["Name"],
        "river": station["River"],
        "COMID_v1": identifier_text(station["COMID_v1"]),
        "COMID_v2": identifier_text(station["COMID_v2"]),
        "hydroweb_file": str(hydroweb_file) if hydroweb_file else "",
        "geoglows_file": str(geoglows_file) if geoglows_file else "",
        "is_loadable": not skip_reasons,
        "skip_reason": "; ".join(skip_reasons),
    })

bulk_station_load_plan = pd.DataFrame(bulk_rows)
loadable_station_plan = bulk_station_load_plan[bulk_station_load_plan["is_loadable"]].copy()

summary = pd.DataFrame([{
    "stations_in_catalog": len(stations_df),
    "hydroweb_station_files": len(hydroweb_index),
    "geoglows_station_files": len(geoglows_index),
    "eligible_stations": int(bulk_station_load_plan["is_loadable"].sum()),
    "stations_selected_for_this_run": len(loadable_station_plan),
}])

display(summary)
display(loadable_station_plan)

,stations_in_catalog,hydroweb_station_files,geoglows_station_files,eligible_stations,stations_selected_for_this_run
0,5,5,5,5,5


,station_id,name,river,COMID_v1,COMID_v2,hydroweb_file,geoglows_file,is_loadable,skip_reason
0,H-102549,Nile_achwa_km5268,Achwa,7068385,160214697,/content/sample_data/ts/hydroweb/H-102549.csv,/content/sample_data/ts/geoglows/160214697.csv,True,
1,H-107870,Nile_akagera_km6130,Akagera,7073854,160220651,/content/sample_data/ts/hydroweb/H-107870.csv,/content/sample_data/ts/geoglows/160220651.csv,True,
2,H-0000000007850,Nile_akagera_km6148,Akagera,7073854,160214812,/content/sample_data/ts/hydroweb/H-00000000078...,/content/sample_data/ts/geoglows/160214812.csv,True,
3,H-101620,Nile_akokoro_km5900,Akokoro,7069648,160214717,/content/sample_data/ts/hydroweb/H-101620.csv,/content/sample_data/ts/geoglows/160214717.csv,True,
4,H-100136,Nile_kafu_km5582,Kafu,7070059,160207720,/content/sample_data/ts/hydroweb/H-100136.csv,/content/sample_data/ts/geoglows/160207720.csv,True,


In [17]:
thing_map = workspace_thing_map()
print({
    station_id: f"{resource_name(thing)} | uuid={resource_uid(thing)}"
    for station_id, thing in thing_map.items()
})


{'H-102549': 'Nile_achwa_km5268 | uuid=019dd257-d6b1-748c-baf3-aed0b33f4cf7', 'H-107870': 'Nile_akagera_km6130 | uuid=019dd257-ddf8-763a-b1ea-8921cf534ea0', 'H-0000000007850': 'Nile_akagera_km6148 | uuid=019dd257-e297-7c6c-82dd-982c6ec14bd4', 'H-101620': 'Nile_akokoro_km5900 | uuid=019dd257-e74b-7724-a9f4-4b3f086026fd', 'H-100136': 'Nile_kafu_km5582 | uuid=019dd257-ebd8-703a-8d04-607a155dc5c2'}


In [18]:
display(workspace_inventory_dataframe())

,resource_type,count,examples
0,things,5,"Nile_achwa_km5268, Nile_akagera_km6130, Nile_a..."
1,datastreams,0,
2,observed_properties,2,"Uganda Demo 20260428042435 Water Level, Uganda..."
3,units,2,"Uganda Demo 20260428042435 Meter, Uganda Demo ..."
4,sensors,2,Uganda Demo 20260428042435 Hydroweb Theia Alti...
5,processing_levels,1,RAW_20260428042435
6,result_qualifiers,1,SUSPECT_20260428042435
7,orchestration_systems,0,
8,data_connections,0,
9,tasks,0,


In [19]:
# Show the current workspace Thing resources as a dataframe.
thing_rows = []
for thing in list_workspace_resources("things"):
    thing_rows.append({
        "station_id": getattr(thing, "sampling_feature_code", None),
        "name": getattr(thing, "name", None),
        "uuid": resource_uid(thing),
    })

display(pd.DataFrame(thing_rows))


,station_id,name,uuid
0,H-102549,Nile_achwa_km5268,019dd257-d6b1-748c-baf3-aed0b33f4cf7
1,H-107870,Nile_akagera_km6130,019dd257-ddf8-763a-b1ea-8921cf534ea0
2,H-0000000007850,Nile_akagera_km6148,019dd257-e297-7c6c-82dd-982c6ec14bd4
3,H-101620,Nile_akokoro_km5900,019dd257-e74b-7724-a9f4-4b3f086026fd
4,H-100136,Nile_kafu_km5582,019dd257-ebd8-703a-8d04-607a155dc5c2


In [20]:
def build_datastream_template(name, description, thing, sensor, observed_property, processing_level, unit, begin_time):
    return {
        "name": name,
        "description": description,
        "observation_type": "Field Observation",
        "sampled_medium": "Water",
        "no_data_value": -9999,
        "aggregation_statistic": "Continuous",
        "time_aggregation_interval": 1,
        "status": "Ongoing",
        "result_type": "Timeseries",
        "value_count": 0,
        "phenomenon_begin_time": begin_time,
        "phenomenon_end_time": None,
        "result_begin_time": begin_time,
        "result_end_time": None,
        "is_visible": True,
        "is_private": False,
        "thing": resource_uid(thing),
        "sensor": resource_uid(sensor),
        "observed_property": resource_uid(observed_property),
        "processing_level": resource_uid(processing_level),
        "unit": resource_uid(unit),
        "time_aggregation_interval_unit": "days",
        "intended_time_spacing": 1,
        "intended_time_spacing_unit": "days",
    }


station_datastreams = []

if hs_api is None or workspace is None or workspace_uid is None:
    print("Skipping creation because an authenticated workspace is required.")
elif loadable_station_plan.empty:
    print("No stations are eligible for bulk loading.")
else:
    try:
        # Build station_id -> Thing directly from the workspace.
        thing_map = workspace_thing_map()
        print(f"Things available in workspace: {len(thing_map)}")

        for _, plan_row in loadable_station_plan.iterrows():
            sid = str(plan_row["station_id"])

            if sid not in thing_map:
                print(f"Skipping {sid}: Thing resource not found in workspace.")
                continue

            current_thing = thing_map[sid]
            station_meta = stations_df[stations_df["ID"].astype(str) == sid].iloc[0]

            # Load and normalize data to get start times.
            hydroweb_payload = normalize_hydroweb_water_level(Path(plan_row["hydroweb_file"]), station_meta)
            geoglows_payload = normalize_two_column_geoglows(Path(plan_row["geoglows_file"]), station_meta)

            hydroweb_template = build_datastream_template(
                name=f"{demo_resource_prefix} {sid} Hydroweb Water Level",
                description="Hydroweb/Theia water-level observations loaded from local CSV.",
                thing=current_thing,
                sensor=hydroweb_sensor,
                observed_property=water_level_observed_property,
                processing_level=processing_level,
                unit=water_level_unit,
                begin_time=hydroweb_payload["phenomenon_time"].min().to_pydatetime(),
            )
            hydroweb_ds = get_or_create_workspace_resource(
                "datastreams",
                {"name": hydroweb_template["name"]},
                hydroweb_template,
                label="datastream",
            )

            geoglows_template = build_datastream_template(
                name=f"{demo_resource_prefix} {sid} GEOGLOWS Streamflow",
                description="GEOGLOWS streamflow observations loaded from local CSV.",
                thing=current_thing,
                sensor=geoglows_sensor,
                observed_property=streamflow_observed_property,
                processing_level=processing_level,
                unit=streamflow_unit,
                begin_time=geoglows_payload["phenomenon_time"].min().to_pydatetime(),
            )
            geoglows_ds = get_or_create_workspace_resource(
                "datastreams",
                {"name": geoglows_template["name"]},
                geoglows_template,
                label="datastream",
            )

            station_datastreams.append({
                "station_id": sid,
                "hydroweb": hydroweb_ds,
                "geoglows": geoglows_ds,
            })

    except Exception as exc:
        print(f"Could not create bulk resources: {exc}")

display(workspace_inventory_dataframe())


Things available in workspace: 5
Created datastream: Uganda Demo 20260428042435 H-102549 Hydroweb Water Level | uuid=019dd259-d4ba-71d3-b5f1-8654a566d210
Created datastream: Uganda Demo 20260428042435 H-102549 GEOGLOWS Streamflow | uuid=019dd259-d960-7555-9485-c7a819b34425
Created datastream: Uganda Demo 20260428042435 H-107870 Hydroweb Water Level | uuid=019dd259-de2a-7a33-b649-c58bc8ba93e9
Created datastream: Uganda Demo 20260428042435 H-107870 GEOGLOWS Streamflow | uuid=019dd259-e6a0-7512-886a-f4918a3e9d89
Created datastream: Uganda Demo 20260428042435 H-0000000007850 Hydroweb Water Level | uuid=019dd259-ef15-7e60-b224-4076263400dd
Created datastream: Uganda Demo 20260428042435 H-0000000007850 GEOGLOWS Streamflow | uuid=019dd259-f3ff-71bf-8095-f66edf7782f9
Created datastream: Uganda Demo 20260428042435 H-101620 Hydroweb Water Level | uuid=019dd259-fbc0-7998-8a31-f3b1deb92db1
Created datastream: Uganda Demo 20260428042435 H-101620 GEOGLOWS Streamflow | uuid=019dd25a-03b1-7df8-8570-a5

,resource_type,count,examples
0,things,5,"Nile_achwa_km5268, Nile_akagera_km6130, Nile_a..."
1,datastreams,10,Uganda Demo 20260428042435 H-102549 Hydroweb W...
2,observed_properties,2,"Uganda Demo 20260428042435 Water Level, Uganda..."
3,units,2,"Uganda Demo 20260428042435 Meter, Uganda Demo ..."
4,sensors,2,Uganda Demo 20260428042435 Hydroweb Theia Alti...
5,processing_levels,1,RAW_20260428042435
6,result_qualifiers,1,SUSPECT_20260428042435
7,orchestration_systems,0,
8,data_connections,0,
9,tasks,0,


# Uploading observation

## Upload observations to HydroServer datastreams

At this point in the notebook, we have already created:

1. A HydroServer workspace.
2. Things/sites for the selected stations.
3. Metadata resources such as observed properties, units, sensors, and processing levels.
4. Datastreams for each station.

Now we will upload the actual time series observations into those datastreams.

Each HydroServer datastream stores one time series. For this workshop notebook, each station may have two datastreams:

- **Hydroweb Water Level** observations
- **GEOGLOWS Streamflow** observations

The upload process has five steps:

1. Read all datastreams from the current HydroServer workspace.
2. Match each station to its Hydroweb and GEOGLOWS datastreams.
3. Read the local observation files.
4. Normalize the observations into the HydroServer format:
   - `phenomenon_time`
   - `result`
5. Upload the observations to HydroServer.

We use `mode="replace"` so the notebook can be run multiple times without duplicating observations. If observations already exist in the datastream, they are replaced with the observations from this notebook.

In [ ]:
# =============================================================================
# Upload observations to HydroServer datastreams
# =============================================================================
#
# This cell uploads time series observations into the datastreams that were
# created earlier in the notebook.
#
# Important idea:
#   We do not use a local "created_resources" dictionary.
#   Instead, we ask HydroServer:
#       "What datastreams exist in this workspace?"
#
# This makes the notebook easier to rerun. Even if the notebook kernel restarts,
# the resources still exist in HydroServer and can be discovered from the
# workspace.
#
# HydroServer expects observations as a pandas DataFrame with at least:
#   - phenomenon_time : timestamp of the observation
#   - result          : numeric observed value
#
# Optional columns can also be included, such as result_qualifier_codes,
# but this workshop cell keeps the upload simple.
# =============================================================================
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
#
# "replace" means:
#   If the datastream already has observations, replace them with the observations
#   from this notebook.
#
# This is useful for workshops because participants can rerun the notebook
# without creating duplicate observations.
#
# To append instead, change this to:
#   UPLOAD_MODE = "append"
#
# In append mode, this cell calls datastream.load_observations(payload)
# without the replace option.
# -----------------------------------------------------------------------------

UPLOAD_MODE = "replace"

# -----------------------------------------------------------------------------
# Main upload workflow
# -----------------------------------------------------------------------------
#
# For each station in loadable_station_plan:
#
#   1. Find the matching station metadata in stations_df.
#   2. Build the expected Hydroweb datastream name.
#   3. Build the expected GEOGLOWS datastream name.
#   4. Find those datastreams in the workspace.
#   5. Read and normalize the local source files.
#   6. Upload the observations.
#
# The result is stored in upload_summary_df so participants can see what happened.
# -----------------------------------------------------------------------------

upload_summary_rows = []

print("Starting HydroServer observation upload...")
print(f"Upload mode: {UPLOAD_MODE}")
print("")

if hs_api is None or workspace_uid is None:
    print("Skipping upload because an authenticated HydroServer workspace is required.")

elif loadable_station_plan.empty:
    print("No loadable stations found.")
    print("Check the previous station loading table: loadable_station_plan")

else:
    datastreams_by_name = workspace_datastream_map()

    print(f"Workspace UUID: {workspace_uid}")
    print(f"Datastreams found in workspace: {len(datastreams_by_name)}")
    print(f"Stations ready for upload: {len(loadable_station_plan)}")
    print("")

    for _, plan_row in loadable_station_plan.iterrows():
        sid = str(plan_row["station_id"])

        print(f"Processing station: {sid}")

        station_matches = stations_df[stations_df["ID"].astype(str) == sid]

        if station_matches.empty:
            message = "Station not found in stations_df."

            print(f"  Skipped: {message}")

            upload_summary_rows.append({
                "station_id": sid,
                "source": "station metadata",
                "datastream": "",
                "status": "skipped",
                "rows_uploaded": 0,
                "begin_time": None,
                "end_time": None,
                "message": message,
            })

            continue

        station_meta = station_matches.iloc[0]

        upload_jobs = [
            {
                "source": "Hydroweb",
                "file": Path(plan_row["hydroweb_file"]),
                "datastream_name": f"{demo_resource_prefix} {sid} Hydroweb Water Level",
                "normalizer": normalize_hydroweb_water_level,
            },
            {
                "source": "GEOGLOWS",
                "file": Path(plan_row["geoglows_file"]),
                "datastream_name": f"{demo_resource_prefix} {sid} GEOGLOWS Streamflow",
                "normalizer": normalize_two_column_geoglows,
            },
        ]

        for job in upload_jobs:
            source_name = job["source"]
            data_file = job["file"]
            datastream_name = job["datastream_name"]

            print(f"  Source: {source_name}")
            print(f"  File: {data_file}")
            print(f"  Target datastream: {datastream_name}")

            datastream = datastreams_by_name.get(datastream_name)

            if datastream is None:
                message = "Datastream not found in workspace."

                print(f"    Skipped: {message}")

                upload_summary_rows.append({
                    "station_id": sid,
                    "source": source_name,
                    "datastream": datastream_name,
                    "status": "skipped",
                    "rows_uploaded": 0,
                    "begin_time": None,
                    "end_time": None,
                    "message": message,
                })

                continue

            if not data_file.exists():
                message = "Source file does not exist."

                print(f"    Skipped: {message}")

                upload_summary_rows.append({
                    "station_id": sid,
                    "source": source_name,
                    "datastream": datastream_name,
                    "status": "skipped",
                    "rows_uploaded": 0,
                    "begin_time": None,
                    "end_time": None,
                    "message": message,
                })

                continue

            try:
                # Convert the source-specific file into the common HydroServer
                # observation format.
                observations_df = job["normalizer"](data_file, station_meta)

                # Upload the normalized observations.
                result = upload_observations_to_datastream(
                    datastream,
                    observations_df,
                    mode=UPLOAD_MODE,
                )

                print(
                    f"    {result['status']}: "
                    f"{result['rows_uploaded']} observations"
                )

                if result.get("begin_time") is not None:
                    print(f"    Begin time: {result['begin_time']}")
                    print(f"    End time:   {result['end_time']}")

                upload_summary_rows.append({
                    "station_id": sid,
                    "source": source_name,
                    "datastream": datastream_name,
                    "status": result["status"],
                    "rows_uploaded": result["rows_uploaded"],
                    "begin_time": result.get("begin_time"),
                    "end_time": result.get("end_time"),
                    "message": result.get("message", ""),
                })

            except Exception as exc:
                message = repr(exc)

                print(f"    Error: {message}")

                upload_summary_rows.append({
                    "station_id": sid,
                    "source": source_name,
                    "datastream": datastream_name,
                    "status": "error",
                    "rows_uploaded": 0,
                    "begin_time": None,
                    "end_time": None,
                    "message": message,
                })

        print("")


# -----------------------------------------------------------------------------
# Display upload summary
# -----------------------------------------------------------------------------
#
# This table is useful for workshop participants because it shows:
#
#   - which station was processed
#   - which source was uploaded
#   - which HydroServer datastream received the observations
#   - how many rows were uploaded
#   - whether there were errors
# -----------------------------------------------------------------------------

upload_summary_df = pd.DataFrame(upload_summary_rows)

print("Upload summary:")
display(upload_summary_df)

Starting HydroServer observation upload...
Upload mode: replace

Workspace UUID: 019dd255-f7ac-7e50-a305-8ffcbdc1d236
Datastreams found in workspace: 10
Stations ready for upload: 5

Processing station: H-102549
  Source: Hydroweb
  File: /content/sample_data/ts/hydroweb/H-102549.csv
  Target datastream: Uganda Demo 20260428042435 H-102549 Hydroweb Water Level
    uploaded: 63 observations
    Begin time: 2020-09-02 00:00:00+00:00
    End time:   2025-06-23 00:00:00+00:00
  Source: GEOGLOWS
  File: /content/sample_data/ts/geoglows/160214697.csv
  Target datastream: Uganda Demo 20260428042435 H-102549 GEOGLOWS Streamflow
    uploaded: 31523 observations
    Begin time: 1940-01-01 00:00:00+00:00
    End time:   2026-04-21 00:00:00+00:00

Processing station: H-107870
  Source: Hydroweb
  File: /content/sample_data/ts/hydroweb/H-107870.csv
  Target datastream: Uganda Demo 20260428042435 H-107870 Hydroweb Water Level
    uploaded: 91 observations
    Begin time: 2018-12-31 00:00:00+00:00
  

Let's look at the first datastream and check



In [ ]:
# Quick verification: fetch observations from the first uploaded datastream.
successful_uploads = upload_summary_df[upload_summary_df["status"] == "uploaded"]

if successful_uploads.empty:
    print("No successful uploads to verify.")
else:
    first_datastream_name = successful_uploads.iloc[0]["datastream"]
    datastream = workspace_datastream_map()[first_datastream_name]

    observations = datastream.get_observations(fetch_all=True).dataframe
    print(first_datastream_name)
    print(f"Rows in HydroServer: {len(observations)}")
    display(observations.head())

## Cleanup: Delete Created Resources for a given workspace

In [ ]:
def clear_hydroserver_workspace(hs_api: Any, workspace_uid: str, *, dry_run: bool = True, delete_workspace: bool = False, continue_on_error: bool = True, page_size: int = 1000,) -> Dict[str, Any]:
    """
    Delete all user-managed HydroServer resources from a workspace.

    WARNING:
        This is destructive when dry_run=False.

    Deletion order:
        1. Task runs, tasks
        2. Data connections
        3. Orchestration systems
        4. Datastream observations
        5. Datastreams
        6. Things/sites
        7. Result qualifiers
        8. Sensors
        9. Units
        10. Processing levels
        11. Observed properties
        12. Optionally the workspace itself

    Parameters
    ----------
    hs_api:
        Authenticated hydroserverpy.HydroServer instance.

    workspace_uid:
        UUID/string ID of the workspace to clear.

    dry_run:
        If True, only reports what would be deleted.

    delete_workspace:
        If True, deletes the workspace after clearing resources.

    continue_on_error:
        If True, continues deleting other resources after a failure.

    page_size:
        Page size for fetch_all list calls.

    Returns
    -------
    dict
        Summary of planned/deleted resources and any errors.
    """

    summary: Dict[str, Any] = {
        "workspace": workspace_uid,
        "dry_run": dry_run,
        "deleted": {},
        "errors": [],
    }

    # Fetch everything first so the dry-run report is complete and the
    # deletion plan is based on a consistent snapshot.
    tasks = _list("tasks")
    dataconnections = _list("dataconnections")
    orchestrationsystems = _list("orchestrationsystems")
    datastreams = _list("datastreams")
    things = _list("things")
    resultqualifiers = _list("resultqualifiers")
    sensors = _list("sensors")
    units = _list("units")
    processinglevels = _list("processinglevels")
    observedproperties = _list("observedproperties")

    # 1. Delete task runs, then tasks.
    for task in tasks:
        try:
            if hasattr(task, "get_task_runs"):
                task_runs_collection = task.get_task_runs()
                task_runs = list(getattr(task_runs_collection, "items", task_runs_collection))

                for task_run in task_runs:
                    try:
                        if not dry_run:
                            task.delete_task_run(uid=getattr(task_run, "uid"))
                        _record_deleted("task_runs", task_run)
                    except Exception as exc:
                        _record_error("task_runs", task_run, exc)

            if not dry_run:
                task.delete()
            _record_deleted("tasks", task)

        except Exception as exc:
            _record_error("tasks", task, exc)

    # 2–3. Delete ETL/orchestration resources before datastreams/metadata.
    _delete_objects("dataconnections", dataconnections)
    _delete_objects("orchestrationsystems", orchestrationsystems)

    # 4. Delete all observations from each datastream.
    for datastream in datastreams:
        try:
            if not dry_run:
                datastream.delete_observations()
            _record_deleted("observations", datastream)
        except Exception as exc:
            _record_error("observations", datastream, exc)

    # 5. Delete datastreams.
    _delete_objects("datastreams", datastreams)

    # 6. Delete things/sites.
    _delete_objects("things", things)

    # 7–11. Delete metadata after datastreams no longer reference it.
    _delete_objects("resultqualifiers", resultqualifiers)
    _delete_objects("sensors", sensors)
    _delete_objects("units", units)
    _delete_objects("processinglevels", processinglevels)
    _delete_objects("observedproperties", observedproperties)

    # 12. Optionally delete the workspace itself.
    if delete_workspace:
        try:
            workspace = hs_api.workspaces.get(uid=workspace_uid)
            if not dry_run:
                workspace.delete()
            _record_deleted("workspaces", workspace)
        except Exception as exc:
            _record_error("workspaces", workspace_uid, exc)

    return summary

In [ ]:
# First: preview what would be deleted
plan = clear_hydroserver_workspace(hs_api, workspace_uid, dry_run=True,)

print(plan)

In [ ]:
result = clear_hydroserver_workspace(hs_api, workspace_uid, dry_run=False, delete_workspace=False,)

print(result)